# Partial Harvard TESSERA embeddings as PCA-RGB

This notebook is safe to run while the embedding job is active. It reads only atomically published `.parquet` shards, verifies each field against its full expected pixel membership, and randomly selects up to three fields that are complete for every currently available window.

Each 128-dimensional embedding is projected onto up to three numerically supported principal components and normalized to RGB; unsupported channels remain neutral. These colors are a diagnostic visualization of embedding similarity—not true-color satellite imagery and not physical spectral bands.

In [ ]:
from collections import defaultdict
from pathlib import Path
import json
import os
import sys

from IPython import get_ipython

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

repo_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "plain_tessera_incremental" / "__init__.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the SpectraJam checkout.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from plain_tessera_incremental.geometry import parse_field_geometry

OUTPUT_DIR = Path(
    os.environ.get(
        "TESSERA_OUTPUT_DIR",
        "/mnt/foundry-az/playground/data/ground_truth/harvard_tessera_incremental",
    )
)
N_FIELDS = 3
RANDOM_SEED = 20260707  # Change this to inspect another reproducible sample.
MAX_PLOT_PIXELS_PER_FIELD_WINDOW = 5000

def read_parquet_file(path, columns=None):
    # ParquetFile reads exactly this file and does not infer hive partitions from its path.
    return pq.ParquetFile(path).read(columns=columns).to_pandas()

print("Output:", OUTPUT_DIR)

In [ ]:
required = ["run.json", "fields.parquet", "field_pixels.parquet"]
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing pipeline artifacts: {missing}")

run = json.loads((OUTPUT_DIR / "run.json").read_text())
completion_path = OUTPUT_DIR / "COMPLETED.json"
run_complete_at_snapshot = completion_path.is_file()
for artifact_name in ("fields", "field_pixels"):
    path = OUTPUT_DIR / f"{artifact_name}.parquet"
    raw_metadata = pq.ParquetFile(path).metadata.metadata or {}
    metadata = {key.decode(): value.decode() for key, value in raw_metadata.items()}
    expected = {
        "run_fingerprint": run["run_fingerprint"],
        "artifact": artifact_name,
        "schema_version": "1",
    }
    mismatched = {
        key: (metadata.get(key), value)
        for key, value in expected.items()
        if metadata.get(key) != value
    }
    if mismatched:
        raise RuntimeError(f"Artifact metadata mismatch for {path}: {mismatched}")

discovered_files = sorted(
    path
    for path in (OUTPUT_DIR / "embeddings").glob("window_id=w*/*.parquet")
    if not path.name.endswith(".part")
)
if not discovered_files:
    raise RuntimeError("No completed embedding shards are available yet.")

required_embedding_columns = {
    "row_id",
    "run_fingerprint",
    "field_uid",
    "pixel_id",
    "window_id",
    "pixel_longitude",
    "pixel_latitude",
    "outcome",
    "embedding",
}
embedding_files = []
seen_task_windows = set()
for path in discovered_files:
    parquet = pq.ParquetFile(path)
    raw_metadata = parquet.metadata.metadata or {}
    metadata = {key.decode(): value.decode() for key, value in raw_metadata.items()}
    path_window = path.parent.name.split("=", 1)[1]
    expected_metadata = {
        "run_fingerprint": run["run_fingerprint"],
        "artifact": "pixel_embeddings",
        "schema_version": "1",
        "window_id": path_window,
    }
    mismatched = {
        key: (metadata.get(key), value)
        for key, value in expected_metadata.items()
        if metadata.get(key) != value
    }
    if mismatched:
        raise RuntimeError(f"Shard metadata mismatch for {path}: {mismatched}")
    if not metadata.get("task_key") or not metadata.get("task_fingerprint"):
        raise RuntimeError(f"Shard task metadata is missing: {path}")
    missing_columns = required_embedding_columns - set(parquet.schema_arrow.names)
    if missing_columns:
        raise RuntimeError(f"Shard {path} is missing columns {sorted(missing_columns)}")
    embedding_type = parquet.schema_arrow.field("embedding").type
    if not pa.types.is_list(embedding_type) or embedding_type.value_type != pa.float32():
        raise RuntimeError(f"Shard {path} has embedding type {embedding_type}")
    task_window = (path_window, metadata.get("task_key"))
    if task_window in seen_task_windows:
        raise RuntimeError(f"Duplicate task/window shard detected: {task_window}")
    seen_task_windows.add(task_window)
    embedding_files.append(path)

windows = sorted(
    {path.parent.name.split("=", 1)[1] for path in embedding_files},
    key=lambda value: int(value.removeprefix("w")),
)
print("Run complete at snapshot:", run_complete_at_snapshot)
print("Published embedding shards:", len(embedding_files))
print("Available windows:", windows)
print("Expected final embedding rows:", run["expected_embedding_rows"])

In [ ]:
fields = read_parquet_file(OUTPUT_DIR / "fields.parquet")
memberships = read_parquet_file(
    OUTPUT_DIR / "field_pixels.parquet",
    columns=["field_uid", "pixel_id"],
)
membership_columns = ["field_uid", "pixel_id"]
if memberships.duplicated(membership_columns).any():
    raise RuntimeError("field_pixels.parquet contains duplicate memberships.")
expected_pair_index = pd.MultiIndex.from_frame(memberships[membership_columns])
expected_pixels = memberships.groupby("field_uid", sort=False).size()

# Stream only the lightweight row index. One boolean per expected membership/window
# detects cross-shard duplicates while avoiding loading every 128-D embedding.
seen_by_window = {
    window_id: np.zeros(len(memberships), dtype=bool) for window_id in windows
}
actual_counts = defaultdict(int)
usable_counts = defaultdict(int)
field_paths = defaultdict(set)

for path in embedding_files:
    path_window = path.parent.name.split("=", 1)[1]
    part = read_parquet_file(
        path,
        columns=["row_id", "field_uid", "pixel_id", "window_id", "outcome"],
    )
    if part.empty:
        raise RuntimeError(f"Published shard is unexpectedly empty: {path}")
    if not part["window_id"].eq(path_window).all():
        raise RuntimeError(f"Shard rows disagree with path window: {path}")
    if not set(part["outcome"].unique()) <= {"complete", "empty_window"}:
        raise RuntimeError(f"Shard has an unknown outcome: {path}")
    actual_pairs = pd.MultiIndex.from_frame(part[membership_columns])
    membership_positions = expected_pair_index.get_indexer(actual_pairs)
    if (membership_positions < 0).any():
        bad = part.loc[membership_positions < 0, membership_columns].head(5)
        raise RuntimeError(
            f"Shard has rows outside field_pixels.parquet: {bad.to_dict('records')}"
        )
    if (
        part["row_id"].duplicated().any()
        or len(np.unique(membership_positions)) != len(membership_positions)
        or seen_by_window[path_window][membership_positions].any()
    ):
        raise RuntimeError(f"Duplicate field/pixel/window row detected in {path}")
    seen_by_window[path_window][membership_positions] = True

    for field_uid, count in part["field_uid"].value_counts().items():
        actual_counts[(field_uid, path_window)] += int(count)
        field_paths[field_uid].add(path)
    for field_uid, count in (
        part.loc[part["outcome"].eq("complete"), "field_uid"].value_counts().items()
    ):
        usable_counts[(field_uid, path_window)] += int(count)

actual_pixels = pd.DataFrame(
    [
        {
            "field_uid": field_uid,
            "window_id": window_id,
            "actual_pixels": count,
            "usable_pixels": usable_counts[(field_uid, window_id)],
        }
        for (field_uid, window_id), count in actual_counts.items()
    ]
)
actual_pixels["expected_pixels"] = actual_pixels["field_uid"].map(expected_pixels)
actual_pixels["field_window_complete"] = (
    actual_pixels["actual_pixels"] == actual_pixels["expected_pixels"]
)

complete_matrix = (
    actual_pixels.pivot(
        index="field_uid", columns="window_id", values="field_window_complete"
    )
    .reindex(columns=windows, fill_value=False)
    .fillna(False)
    .astype(bool)
)
usable_matrix = (
    actual_pixels.pivot(index="field_uid", columns="window_id", values="usable_pixels")
    .reindex(index=complete_matrix.index, columns=windows, fill_value=0)
    .fillna(0)
    .astype(int)
)

fully_available = complete_matrix.all(axis=1)
eligible = fully_available & usable_matrix.gt(0).all(axis=1)
if eligible.sum() < N_FIELDS:
    # During an early partial run, allow a completed field with at least one usable window.
    eligible = fully_available & usable_matrix.sum(axis=1).gt(0)

eligible_fields = eligible[eligible].index.to_numpy()
if len(eligible_fields) == 0:
    raise RuntimeError(
        "No field is fully published across the currently available windows yet. "
        "Wait for more completed tasks and rerun this cell."
    )
sample_size = min(N_FIELDS, len(eligible_fields))
rng = np.random.default_rng(RANDOM_SEED)
selected_fields = rng.choice(eligible_fields, size=sample_size, replace=False).tolist()

window_summary = actual_pixels.groupby("window_id").agg(
    published_field_windows=("field_uid", "nunique"),
    fully_complete_field_windows=("field_window_complete", "sum"),
)
display(window_summary.reindex(windows))
display(
    fields.set_index("field_uid")
    .loc[selected_fields, ["id", "landcover", "pixel_count", "coordinate_status"]]
)

In [ ]:
index_columns = [
    "row_id",
    "field_uid",
    "pixel_id",
    "window_id",
    "pixel_longitude",
    "pixel_latitude",
    "outcome",
]
selected_paths = sorted(
    {path for field_uid in selected_fields for path in field_paths[field_uid]},
    key=str,
)
selected_index_parts = []
for path in selected_paths:
    part = pq.read_table(
        path,
        columns=index_columns,
        filters=[("field_uid", "in", selected_fields)],
        partitioning=None,
    ).to_pandas()
    if not part.empty:
        part["_path"] = str(path)
        selected_index_parts.append(part)
if not selected_index_parts:
    raise RuntimeError("Selected fields disappeared from the published shard snapshot.")

selected_index = pd.concat(selected_index_parts, ignore_index=True)
if (
    selected_index["row_id"].duplicated().any()
    or selected_index.duplicated(["field_uid", "pixel_id", "window_id"]).any()
):
    raise RuntimeError("Selected field rows contain duplicate memberships.")

# Keep the preview bounded for unusually large fields. Sampling is deterministic
# and always retains at least one usable row when the group contains one.
sampled_indices = []
for _, group in selected_index.groupby(["field_uid", "window_id"], sort=False):
    ordered = group.sort_values("pixel_id")
    if len(ordered) <= MAX_PLOT_PIXELS_PER_FIELD_WINDOW:
        chosen = ordered
    else:
        positions = np.linspace(
            0, len(ordered) - 1, MAX_PLOT_PIXELS_PER_FIELD_WINDOW, dtype=np.int64
        )
        chosen = ordered.iloc[positions]
        complete = ordered[ordered["outcome"].eq("complete")]
        if not complete.empty and not chosen["outcome"].eq("complete").any():
            chosen = pd.concat([complete.iloc[[0]], chosen.iloc[1:]])
    sampled_indices.extend(chosen.index.tolist())
selected_rows = selected_index.loc[sampled_indices].copy().reset_index(drop=True)

# Fetch the heavy list<float32>[128] column only for the sampled row IDs.
embedding_parts = []
for path_text, rows_for_path in selected_rows.groupby("_path", sort=False):
    row_ids = rows_for_path["row_id"].astype(str).tolist()
    part = pq.read_table(
        Path(path_text),
        columns=["row_id", "embedding"],
        filters=[("row_id", "in", row_ids)],
        partitioning=None,
    ).to_pandas()
    embedding_parts.append(part)
embedding_rows = pd.concat(embedding_parts, ignore_index=True)
if embedding_rows["row_id"].duplicated().any():
    raise RuntimeError("Filtered embedding read returned duplicate row IDs.")
selected_rows = selected_rows.merge(
    embedding_rows, on="row_id", how="left", validate="one_to_one", indicator=True
)
if not selected_rows["_merge"].eq("both").all():
    raise RuntimeError("Filtered embedding read did not return every sampled row.")
selected_rows = selected_rows.drop(columns=["_merge"])

def embedding_vector(value):
    if value is None:
        return None
    vector = np.asarray(value, dtype=np.float32)
    if vector.shape != (128,) or not np.isfinite(vector).all():
        return None
    return vector

selected_rows["_vector"] = selected_rows["embedding"].map(embedding_vector)
usable = selected_rows["outcome"].eq("complete") & selected_rows["_vector"].notna()
pca_rows = selected_rows.loc[usable].drop_duplicates(["pixel_id", "window_id"])
if len(pca_rows) < 3:
    raise RuntimeError("Fewer than three finite physical pixel embeddings are available.")

matrix = np.stack(pca_rows["_vector"].to_numpy())
centered = matrix - matrix.mean(axis=0, keepdims=True)
_, singular_values, right_vectors = np.linalg.svd(centered, full_matrices=False)
rank_tolerance = (
    np.finfo(matrix.dtype).eps * max(centered.shape) * singular_values[0]
)
component_count = min(3, int(np.count_nonzero(singular_values > rank_tolerance)))
if component_count == 0:
    raise RuntimeError("The sampled embeddings have no finite PCA variation.")
scores = centered @ right_vectors[:component_count].T

low = np.quantile(scores, 0.02, axis=0)
high = np.quantile(scores, 0.98, axis=0)
spread = np.where(high > low, high - low, 1.0)
scaled_scores = np.clip((scores - low) / spread, 0.0, 1.0)
rgb = np.full((len(scores), 3), 0.5, dtype=np.float32)
rgb[:, :component_count] = scaled_scores
rgb_by_pixel_window = {
    (pixel_id, window_id): color
    for pixel_id, window_id, color in zip(
        pca_rows["pixel_id"], pca_rows["window_id"], rgb, strict=True
    )
}
selected_rows["rgb"] = [
    rgb_by_pixel_window.get((pixel_id, window_id)) if row_usable else None
    for pixel_id, window_id, row_usable in zip(
        selected_rows["pixel_id"], selected_rows["window_id"], usable, strict=True
    )
]

display(
    selected_rows.groupby(["field_uid", "window_id", "outcome"])
    .size()
    .rename("preview_rows")
    .to_frame()
)

In [ ]:
def polygon_parts(geometry):
    if geometry.geom_type == "Polygon":
        yield geometry
    elif hasattr(geometry, "geoms"):
        for child in geometry.geoms:
            yield from polygon_parts(child)

def draw_field_outline(axis, geometry):
    for polygon in polygon_parts(geometry):
        x, y = polygon.exterior.xy
        axis.plot(x, y, color="black", linewidth=1.2, zorder=3)
        for interior in polygon.interiors:
            x, y = interior.xy
            axis.plot(x, y, color="black", linewidth=0.8, zorder=3)

field_lookup = fields.set_index("field_uid")
figure, axes = plt.subplots(
    len(selected_fields),
    len(windows),
    figsize=(4.2 * len(windows), 4.0 * len(selected_fields)),
    squeeze=False,
)

for row_number, field_uid in enumerate(selected_fields):
    metadata = field_lookup.loc[field_uid]
    geometry, _ = parse_field_geometry(str(metadata["wkt"]))
    min_x, min_y, max_x, max_y = geometry.bounds
    pad_x = max((max_x - min_x) * 0.05, 1e-5)
    pad_y = max((max_y - min_y) * 0.05, 1e-5)

    for column_number, window_id in enumerate(windows):
        axis = axes[row_number, column_number]
        draw_field_outline(axis, geometry)
        rows = selected_rows[
            selected_rows["field_uid"].eq(field_uid)
            & selected_rows["window_id"].eq(window_id)
        ]
        colored = rows[rows["rgb"].notna()]
        empty = rows[rows["rgb"].isna()]
        if not colored.empty:
            colors = np.stack(colored["rgb"].to_numpy())
            axis.scatter(
                colored["pixel_longitude"],
                colored["pixel_latitude"],
                c=colors,
                marker="s",
                s=38,
                linewidths=0,
                zorder=2,
            )
        if not empty.empty:
            axis.scatter(
                empty["pixel_longitude"],
                empty["pixel_latitude"],
                c="#bdbdbd",
                marker="x",
                s=22,
                linewidths=0.8,
                zorder=1,
            )
        axis.set_xlim(min_x - pad_x, max_x + pad_x)
        axis.set_ylim(min_y - pad_y, max_y + pad_y)
        axis.set_aspect("equal", adjustable="box")
        published = int(expected_pixels.loc[field_uid])
        sampled = " (sampled)" if len(rows) < published else ""
        axis.set_title(
            f"{window_id}: {len(colored)} RGB / {len(rows)} shown{sampled}\n"
            f"{published} published pixels"
        )
        axis.tick_params(labelsize=7)
        if column_number == 0:
            axis.set_ylabel(
                f"id={metadata['id']}\n{metadata['landcover']}",
                fontsize=9,
            )

figure.suptitle(
    "Random fully published fields — 128-D TESSERA embeddings projected to PCA-RGB",
    fontsize=14,
    y=1.01,
)
figure.tight_layout()
plt.show()

## Reading the plot

- Similar RGB colors indicate similar locations in the three-component PCA projection.
- Gray × markers are retained pixel/window rows with no embedding (`empty_window`).
- A color change across columns shows how that pixel's embedding changes as the cumulative temporal window grows.
- Change `RANDOM_SEED` and rerun from the selection cell to inspect another random set of completed fields.
- For large fields, each field/window preview is deterministically capped by `MAX_PLOT_PIXELS_PER_FIELD_WINDOW`; the title marks sampled panels.
- PCA is fitted once to unique physical `(pixel_id, window_id)` vectors in the selected preview, so overlapping field memberships do not receive extra weight.
- If the sampled embeddings have fewer than three supported principal components, unused RGB channels stay at a neutral value instead of amplifying numerical noise.
- Colors are comparable across panels in this execution, but not across separate notebook runs unless the selected fields and preview rows are unchanged.